In [1]:
import os
import sys
from typing import List, Tuple
import system_messages
from utils.dataset import SatDataset, custom_collate
from utils.data_analysis import log_data_sample
from torch.utils.data import DataLoader
from dotmap import DotMap
from tqdm import tqdm
import torch
import vertexai
from vertexai.preview.language_models import TextGenerationModel

os.environ["ROOT_PATH"] = os.getcwd()
sys.path.append(os.environ["ROOT_PATH"])
! gcloud config set project 'sat-solving'
! gcloud auth application-default login

Updated property [core/project].


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update



To take a quick anonymous survey, run:
  $ gcloud survey

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=4a5iLuP8rOw35EUJ3t3qyf5yoiYYJW&access_type=offline&code_challenge=6hhwWxP6IOatbQIa5R9oWww4WH3n_DGeP5e__N3lsjc&code_challenge_method=S256


Credentials saved to file: [/home/rishihazra/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "sat-so

In [12]:
import google.auth
creds, _ = google.auth.default(quota_project_id='sat-solving')
vertexai.init(project='sat-solving', credentials=creds)
from google.api_core.exceptions import InvalidArgument

In [13]:
# os.system("auth.authenticate_user()")

def query_palm(model_name: str, preferences: str) -> Tuple[str, int, int]:
    """
    :param model_name: gpt-4, gpt-3.5
    :param preferences: food preferences of people
    :return: raw output generated by gpt-* model, # prompt_tokens, # completion_tokens
    """
    model_name = "text-bison@002"
    generation_model = TextGenerationModel.from_pretrained(model_name)
    
    PROMPT = f"{system_message}\n{preferences}"
    num_prompt_tokens = generation_model.count_tokens(system_message).total_tokens
    num_prompt_tokens += generation_model.count_tokens(preferences).total_tokens

    response = generation_model.predict(
        prompt=PROMPT,
        temperature=1,
        max_output_tokens=1024,
        top_p=1
    )
    
    num_complete_tokens = generation_model.count_tokens(response.text).total_tokens
    
    return (response.text, num_prompt_tokens, num_complete_tokens)


if __name__ == "__main__":
    ablation = 'translate'  # 'menu', 'sat', 'translate'
    # system message to prompt the model
    # different system messages for different ablations
    system_message = system_messages.names[ablation]
    model_name = 'text-bison@002'  # gpt-4, gpt-3.5, llama-2-70b, llama-2-13b, text-bison@002
    data_path = os.path.join(os.environ["ROOT_PATH"], 'dataset.pkl')
    sat_dataset = SatDataset(root_path=os.environ["ROOT_PATH"], data_path=data_path)

    batch_size = 1
    sat_loader = DataLoader(sat_dataset, batch_size=batch_size, shuffle=True, collate_fn=custom_collate)

    for data_sample in tqdm(sat_loader):
        data_sample = [DotMap(sample) for sample in data_sample]
        data_input = data_sample[0].preferences if ablation in ['menu', 'translate'] \
            else data_sample[0].formula
        try:
            gen_out, num_prompt_tokens, num_completion_tokens = query_palm(model_name, data_input)
            log_data_sample(model_name, data_sample[0].num_vars, data_sample[0].num_clauses,
                            data_sample[0].formula, data_sample[0].is_sat,
                            data_sample[0].preferences, data_sample[0].menu_items,
                            gen_out, num_prompt_tokens, num_completion_tokens,
                            ablation=ablation)
        except InvalidArgument as e:
            continue

  0%|                                                                                                                                                                                                                                                                            | 0/24000 [08:02<?, ?it/s]


InternalServerError: 500 Internal error encountered.